In [1]:
## Quantum Computing imports
import pennylane as qml
import numpy as np
from math import pi

## GNN imports
import torch
from torch_geometric.data import Data, DataLoader
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, global_mean_pool, EdgeConv
from sklearn.metrics import accuracy_score, roc_auc_score

## Plotting imports
import matplotlib.pyplot as plt

import os
import random
import notebook_tools as nbtools

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
## Load in the files
files = ["QG_jets.npz"]
X_list, y_list = [], []

for f in files:
    data = np.load(f)
    X_list.append(data["X"])
    y_list.append(data["y"])

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

print("Shape of X:", X.shape)  ## (num_jets, num_particles, num_features)
print("Shape of y:", y.shape)  ## (num_jets,)

Shape of X: (100000, 139, 4)
Shape of y: (100000,)


In [4]:
graph_list = []
k = 16

for i in range(X.shape[0]):
    particles = torch.tensor(X[i], dtype=torch.float)

    ## Preprocess ata
    node_features, eta, phi = nbtools.preprocess(particles)

    ## Skip empty jets
    if node_features.shape[0] < k:
        continue

    ## Build graph
    edge_index = nbtools.build_edge_index(eta, phi, k)


    ## Get edge features
    edge_attr = nbtools.build_edge_features(edge_index, eta, phi)

    label = torch.tensor(y[i], dtype=torch.long)

    graph = Data(
        x=node_features,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=label
    )

    graph_list.append(graph)


/Users/alexcampbell/Documents/QGNN-HybridGNN-QNN/notebook_tools.py:46: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  pt  = (pt - pt.mean()) / (pt.std() + 1e-6)
/Users/alexcampbell/Documents/QGNN-HybridGNN-QNN/notebook_tools.py:47: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  eta = (eta - eta.mean()) / (eta.std() + 1e-6)


In [5]:
# ✅ Shuffle dataset first (VERY important)
random.Random(42).shuffle(graph_list)

n = len(graph_list)

train_end = int(0.7 * n)
val_end   = int(0.85 * n)

train_data = graph_list[:train_end]
val_data   = graph_list[train_end:val_end]
test_data  = graph_list[val_end:]

# ✅ Create loaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32)
test_loader  = DataLoader(test_data, batch_size=32)

/var/folders/5k/_fb_j44j3fxbf0t_zh6q6xkr0000gn/T/ipykernel_27623/3052091688.py:14: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
/var/folders/5k/_fb_j44j3fxbf0t_zh6q6xkr0000gn/T/ipykernel_27623/3052091688.py:15: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader   = DataLoader(val_data, batch_size=32)
/var/folders/5k/_fb_j44j3fxbf0t_zh6q6xkr0000gn/T/ipykernel_27623/3052091688.py:16: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader  = DataLoader(test_data, batch_size=32)


In [ ]:
class JetGNN(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, 64)
        self.conv2 = SAGEConv(64, 64)
        self.lin = torch.nn.Linear(64, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)

model = JetGNN(in_channels=X.shape[2])
model = nbtools.train_model(
    model,
    train_loader,
    val_loader=val_loader,   # or split a validation set
    epochs=10,
    patience=5,
    save_every=1
)
acc, auc = nbtools.evaluate_model(model, test_loader)

In [ ]:
class ParticleNet(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super().__init__()
        self.conv1 = EdgeConv(torch.nn.Sequential(
            torch.nn.Linear(2*in_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels, hidden_channels)
        ))
        self.conv2 = EdgeConv(torch.nn.Sequential(
            torch.nn.Linear(2*hidden_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels, hidden_channels)
        ))
        self.conv3 = EdgeConv(torch.nn.Sequential(
            torch.nn.Linear(2*hidden_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels, hidden_channels)
        ))
        self.fc1 = torch.nn.Linear(hidden_channels, 64)
        self.fc2 = torch.nn.Linear(64, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))

        x = global_mean_pool(x, batch)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = ParticleNet(in_channels=X.shape[2])
model = nbtools.train_model(
    model,
    train_loader,
    val_loader=val_loader,   # or split a validation set
    epochs=10,
    patience=5,
    save_every=1
)
acc, auc = nbtools.evaluate_model(model, test_loader)

In [ ]:
n_qubits = 8   
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnn_circuit_basic(x, weights):
    # x shape expected: (batch_size, n_qubits)
    
    ## Vectorized Encoding
    # AngleEmbedding natively supports batched inputs, replacing your RY loop
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Y')

    ## Variational Layers (unchanged)
    for l in range(weights.shape[0]):
        ## Entanglement Ring
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i+1) % n_qubits])

        ## Apply Rotations
        for i in range(n_qubits):
            # weights[l, i] is a scalar, but PennyLane handles the broadcasting 
            # alongside the batched 'x' inputs seamlessly.
            qml.RY(weights[l, i], wires=i)

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [ ]:
n_qubits = 8
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnn_circuit_improved(x, weights):
    """
    x: shape (batch_size, n_qubits)
    weights: shape (n_layers, n_qubits, 3)
    """

    n_layers = weights.shape[0]

    for l in range(n_layers):

        ## 🔁 DATA RE-UPLOADING (every layer)
        qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Y')

        ## 🔧 EXPRESSIVE TRAINABLE BLOCK
        for i in range(n_qubits):
            qml.RX(weights[l, i, 0], wires=i)
            qml.RY(weights[l, i, 1], wires=i)
            qml.RZ(weights[l, i, 2], wires=i)

        ## 🔗 STRONGER ENTANGLEMENT (ring + skip)
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i+1) % n_qubits])

        # add longer-range correlations
        for i in range(0, n_qubits, 2):
            qml.CNOT(wires=[i, (i+2) % n_qubits])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [ ]:
class HybridGNN_QNN_basic(torch.nn.Module):
    def __init__(self, in_channels, hidden_dim=64, n_qubits=8, q_layers=4):
        super().__init__()
        self.n_qubits = n_qubits

        ## GNN Backbone
        self.conv1 = SAGEConv(in_channels, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.lin_proj = torch.nn.Linear(hidden_dim, n_qubits)

        ## QNN Params
        self.q_weights = torch.nn.Parameter(
            0.01 * torch.randn(q_layers, n_qubits, 3)
        )

        ## Final Classifier
        self.fc = torch.nn.Linear(n_qubits, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.lin_proj(x)
        x = torch.tanh(x) * torch.pi

        # x is now shape (batch_size, n_qubits)
        device = x.device

        # --- VECTORIZED QNN CALL ---
        # No more list comprehension! Pass the whole batch.
        q_out = qnn_circuit_basic(x, self.q_weights) 

        # q_out is a tuple of length 8. Each element has shape (batch_size,)
        # Stack along dim=1 to reconstruct the (batch_size, 8) tensor
        x = torch.stack(q_out, dim=1).to(device).float()

        return self.fc(x)
    
model = HybridGNN_QNN_basic(in_channels=4)

model = nbtools.train_model(
    model,
    train_loader,
    val_loader=val_loader,   # or split a validation set
    epochs=10,
    patience=5,
    save_every=1
)
acc, auc = nbtools.evaluate_model(model, test_loader)


In [ ]:
class HybridGNN_QNN_improved(torch.nn.Module):
    def __init__(self, in_channels, hidden_dim=64, n_qubits=8, q_layers=4):
        super().__init__()
        self.n_qubits = n_qubits

        ## GNN Backbone
        self.conv1 = SAGEConv(in_channels, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.lin_proj = torch.nn.Linear(hidden_dim, n_qubits)

        ## QNN Params
        self.q_weights = torch.nn.Parameter(
            0.01 * torch.randn(q_layers, n_qubits, 3)
        )

        ## Final Classifier
        self.fc = torch.nn.Linear(n_qubits, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.lin_proj(x)
        x = torch.tanh(x) * torch.pi

        # x is now shape (batch_size, n_qubits)
        device = x.device

        # --- VECTORIZED QNN CALL ---
        # No more list comprehension! Pass the whole batch.
        q_out = qnn_circuit_improved(x, self.q_weights) 

        # q_out is a tuple of length 8. Each element has shape (batch_size,)
        # Stack along dim=1 to reconstruct the (batch_size, 8) tensor
        x = torch.stack(q_out, dim=1).to(device).float()

        return self.fc(x)
    
model = HybridGNN_QNN_improved(in_channels=4)

model = nbtools.train_model(
    model,
    train_loader,
    val_loader=val_loader,   # or split a validation set
    epochs=10,
    patience=5,
    save_every=1
)
acc, auc = nbtools.evaluate_model(model, test_loader)
